# Job Finder — Data Preparation

**ISA632 Group Project | Phase 1 of 2**

This notebook covers the first half of the RAG pipeline: ingesting raw job listing documents from a Unity Catalog Volume, parsing them using Databricks' built-in AI function, loading the results into a Delta table, and verifying the Vector Search index is ready for retrieval.

**Run order:**
1. `01_DataPreparation.ipynb` ← you are here
2. `02_AgentDevelopment.ipynb` — build, register, and deploy the RAG agent

**Prerequisites:**
- Unity Catalog volume at `/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/` containing job listing PDFs
- A Vector Search endpoint named `vs_endpoint_boopatt` already created in Databricks
- Catalog: `isa632_7474656346303369` | Schema: `boopatt`

## Step 1 — Install Dependencies

Install all required packages for the data pipeline. The kernel restarts automatically after installation to load the new libraries.

**After the restart, re-run all cells from Step 2 onward — do not skip any cells.**

In [ ]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0

dbutils.library.restartPython()

## Step 2 — Explore Source Documents

List all files available in the Unity Catalog Volume to confirm the job listing PDFs are accessible before parsing.

In [ ]:
%sql
SELECT path
FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile')

## Step 3 — Parse Documents with `ai_parse_document()`

`ai_parse_document()` is a built-in Databricks SQL function (available on DBR 17.1+ or Serverless) that extracts structured content from binary files such as PDFs. It returns a JSON object containing the document's text elements.

Run the cell below first to inspect the raw parsed output structure before extracting the text content.

In [ ]:
%sql
-- Preview the raw ai_parse_document output on a single file to understand the structure
-- LIMIT 1 avoids parsing all 30 PDFs just for inspection
SELECT ai_parse_document(content) AS parsed_document
FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile')
LIMIT 1

Now extract the plain text content by flattening the document elements array and joining them into a single string per document. The `doc_uri` column records the source file path for traceability.

In [ ]:
%sql
-- Preview extracted text content from a single document before full ingestion
-- LIMIT 1 avoids parsing all 30 PDFs just for inspection
SELECT
  array_join(
    transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content),
    '\n'
  ) AS content,
  path AS doc_uri
FROM (
  SELECT ai_parse_document(content) AS parsed_document, path
  FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile')
  LIMIT 1
)

## Step 4 — Create the Knowledge Base Delta Table

Create the `jobs_knowledge_base` Delta table with **Change Data Feed** enabled. Change Data Feed is required by Databricks Vector Search to automatically detect and sync any inserts, updates, or deletes from this table into the vector index.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS isa632_7474656346303369.boopatt.jobs_knowledge_base (
  id      BIGINT GENERATED ALWAYS AS IDENTITY,
  content STRING,
  doc_uri STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

## Step 5 — Populate the Knowledge Base

Parse all documents from the Volume and insert them into the knowledge base table. `INSERT OVERWRITE` is used so re-running this cell replaces stale data with fresh content.

In [ ]:
%sql
INSERT OVERWRITE TABLE isa632_7474656346303369.boopatt.jobs_knowledge_base (content, doc_uri)
SELECT content, doc_uri
FROM (
  SELECT
    array_join(
      transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content),
      '\n'
    ) AS content,
    path AS doc_uri
  FROM (
    SELECT ai_parse_document(content) AS parsed_document, path
    FROM READ_FILES('/Volumes/isa632_7474656346303369/default/isa632_boopatt/project/', format => 'binaryFile')
  )
)
WHERE content IS NOT NULL AND trim(content) != '';

## Step 6 — Sync the Vector Search Index

After populating the knowledge base table, trigger a sync in the Databricks UI to update the Vector Search index:

1. Go to **Catalog** → `isa632_7474656346303369` → `boopatt` → `jobindex`
2. Click **Sync** to push the latest data from `jobs_knowledge_base` into the index
3. Wait for the sync to complete before running the next step

The index name is: `isa632_7474656346303369.boopatt.jobindex`

## Step 7 — Verify Vector Search Retrieval

Run a quick similarity search to confirm the Vector Search index is populated and returning relevant results. If this fails, check that the sync in Step 6 completed successfully.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)
question = "data analyst job"

try:
    index = vsc.get_index("vs_endpoint_boopatt", "isa632_7474656346303369.boopatt.jobindex")
    results = index.similarity_search(
        query_text=question,
        columns=["content"],
        num_results=4
    )
    docs = results.get("result", {}).get("data_array", [])
    print(f"Retrieved {len(docs)} result(s) for query: '{question}'")
    for i, doc in enumerate(docs, 1):
        print(f"\n--- Result {i} (first 300 chars) ---")
        print(str(doc[0])[:300])
except Exception as e:
    print(f"Error: {e}\nMake sure the Vector Search index sync in Step 6 is complete.")

---

**Data preparation complete.** The knowledge base table is populated and the Vector Search index is verified.

**Next:** Open `02_AgentDevelopment.ipynb` to build, register, and deploy the RAG agent.